In [4]:
import torch
import numpy as np
import random
import os
from amr.utils import logger
from amr.models import *
import importlib
from torchsummaryX import summary


__all__ = ["init_device", "init_model", "init_loss"]


def init_device(seed=None, cpu=None, gpu=None):
    '''
    配置计算设备和随机种子
    '''
    if seed is not None:
        # 设置随机种子：确保实验结果可复现
        # 对Python内置随机数生成器、PyTorch和NumPy都设置相同的种子
        random.seed(seed)
        torch.manual_seed(seed)
        np.random.seed(seed)
        # 设置cudnn.deterministic为True以确保CUDA操作的结果一致
        torch.backends.cudnn.deterministic = True

    # 如果指定了GPU编号，设置CUDA可见设备
    # 这允许程序只使用指定的GPU
    if gpu is not None:
        os.environ['CUDA_VISIBLE_DEVICES'] = str(gpu)

    # 设备选择逻辑
    # 如果未指定使用CPU且CUDA可用，则使用GPU
    if not cpu and torch.cuda.is_available():
        # 设置设备为CUDA
        device = torch.device('cuda')

        # 启用cuDNN自动调优以寻找最佳卷积算法
        # 注意：这可能会导致结果略有不同，但通常会提高性能
        torch.backends.cudnn.benchmark = True

        # 如果设置了种子，为CUDA操作设置随机种子
        if seed is not None:
            torch.cuda.manual_seed(seed)

        # pin_memory=True：加速CPU到GPU的数据传输
        pin_memory = True

        # 记录日志：当前使用的GPU编号
        logger.info("Running on GPU%d" % (gpu if gpu else 0))

    else:
        # 如果使用CPU或CUDA不可用，则使用CPU
        pin_memory = False
        device = torch.device('cpu')
        logger.info("Running on CPU")

    # 返回设备对象和pin_memory标志
    return device, pin_memory


def init_model(args, network):
    '''
    初始化模型并加载预训练权重（如果处于测试模式）
    '''

    # importlib.import_module("amr.models.networks." + args.method + '.' + network)：导入模块文件
    # getattr(, network)：获取网络类
    # …(len(args.mod_type))：实例化类，args.mod_type为调制类型数组
    model = getattr(importlib.import_module("amr.models.networks." + args.method + '.' + network), network)(
        len(args.mod_type))

    # 打印模型结构
    print(model)

    # 调用模块中的测试函数
    getattr(importlib.import_module("amr.models.networks." + args.method + '.' + network), "test")()

    # 非训练模式，从保存的最佳模型文件中加载模型
    if not args.train:

        pretrained = 'results/' + args.method + '/' + network + '/' + args.dataset + '/checkpoints/best_acc.pth'

        # 确保最佳模型文件存在
        assert os.path.isfile(pretrained)

        # torch.load()：加载文件中的内容，即状态字典state
        # 将状态字典state中的键为'state_dict'的值赋值给state_dict
        state_dict = torch.load(pretrained, map_location=torch.device('cpu'))['state_dict']

        # 将状态字典恢复到模型中
        model.load_state_dict(state_dict)

        logger.info("pretrained model loaded from {}".format(pretrained))

    return model


def init_loss(loss_func):
    # importlib.import_module("amr.models.losses." + loss_func)：动态导入模块文件
    # getattr( , loss_func)：获取模块中的对应的类
    # loss = ……()：实例化类
    loss = getattr(importlib.import_module("amr.models.losses." + loss_func), loss_func)()
    return loss


if __name__ == '__main__':
    getattr(importlib.import_module("amr.models.networks.CM_ResNet.CM_ResNet"), "test")()

flops & params ['26.103M', '328.395K']
